In [8]:
import os
import dotenv

dotenv.load_dotenv(".env")



True

In [9]:
from dx_client import DXClient
from dx_client.dx_models import DXClientConfig

client = DXClient(
    config=DXClientConfig(
        auth_token=os.getenv("DX_AUTH_TOKEN", ""),
        project_context_id=os.getenv("DX_PROJECT_CONTEXT_ID", ""),
    )
)
client.connect()
print(f"Connected: {client.is_connected}")


Connected: True


In [9]:
client.list_cohorts()

[DXRecordInfo(id='record-J7KqzGQJj70Z1KzX9j65FVPV', name='Hypertension Olink Study Cohort', project='project-J3Y7p9jJj70gKV8XKv2yxqv5', folder='/', created=1776140210000, modified=1776140609760, state='closed', description='', properties={}, types=['DatabaseQuery', 'CohortBrowser'], details={}, class='record', sponsored=False, hidden=False, links=['record-J77PBFjJVpqqvFz65XvV0Kk6'], tags=[], createdBy={'user': 'user-jiehua_li'}, size=1467)]

In [10]:
client.get_cohort("record-J7Pj7YQJj70q60213Gx10Gbb")

DXRecordInfo(id='record-J7Pj7YQJj70q60213Gx10Gbb', name='PWAS HTN Case-Control v3', project='project-J3Y7p9jJj70gKV8XKv2yxqv5', folder='/', created=1776255950000, modified=1776255955766, state='closed', description='', properties={}, types=['DatabaseQuery', 'CohortBrowser'], details={'databases': ['database-J77JYVjJVpqqx189JkJb87qG', 'database-Fx7Fj58JjG8Yf6bKPX9Yzykf'], 'dataset': {'$dnanexus_link': 'record-J77PBFjJVpqqvFz65XvV0Kk6'}, 'description': 'Hypertension PWAS cohort. Case: p131286 not empty (I10 essential HTN). Control: p131286 empty. All participants must have Olink proteomics data and complete covariates.', 'filters': {'logic': 'and', 'pheno_filters': {'logic': 'and', 'compound': [{'name': 'phenotype', 'logic': 'and', 'filters': {'olink_instance_0$eid': [{'condition': 'exists'}], 'participant$p31': [{'condition': 'exists'}], 'participant$p21003_i0': [{'condition': 'exists'}], 'participant$p21001_i0': [{'condition': 'exists'}], 'participant$p1239_i0': [{'condition': 'exists'

In [12]:
df = client.download_cohort("record-J7Pj5Q0Jj70pZ16zQX7G9v9j")

In [ ]:
df

,participant.eid,participant.p131286,participant.p31,participant.p21003_i0,participant.p21001_i0,participant.p1239_i0,participant.p1249_i0,participant.p20117_i0,participant.p6138_i0,participant.p738_i0,participant.p22189,participant.p54_i0,participant.p4080_i0_a0,participant.p4079_i0_a0,participant.p4080_i0_a1,participant.p4079_i0_a1,participant.p2966_i0,participant.p20161_i0,olink_instance_0.eid
0,1000128,2003-07-01,0,62,27.917244,0.0,4.0,2.0,"[2,3]",2.0,4.84,11020,127.0,70.0,113.0,74.0,55.0,NaN,NaN
1,1000240,NaN,0,43,19.838533,1.0,NaN,0.0,[1],1.0,1.21,11013,99.0,66.0,93.0,64.0,NaN,9.275,NaN
2,1000401,NaN,1,56,20.532472,0.0,4.0,2.0,[1],4.0,1.08,11002,110.0,70.0,108.0,75.0,NaN,NaN,NaN
3,1000477,2001-09-01,0,62,28.425545,0.0,2.0,0.0,[3],2.0,0.49,11011,138.0,77.0,130.0,78.0,55.0,NaN,NaN
4,1000506,NaN,0,63,23.662551,0.0,4.0,2.0,[3],1.0,-2.14,11016,153.0,80.0,135.0,81.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501931,6020670,NaN,1,64,26.939896,0.0,1.0,2.0,[5],4.0,-2.92,11014,174.0,99.0,162.0,83.0,NaN,27.500,6020670
501932,6020762,NaN,1,43,26.937618,2.0,2.0,2.0,"[3,6]",3.0,0.13,11011,139.0,82.0,144.0,94.0,NaN,NaN,NaN
501933,6020998,NaN,0,40,25.476660,0.0,1.0,2.0,"[1,2,3,6]",5.0,-5.21,11003,123.0,81.0,127.0,80.0,NaN,2.000,NaN
501934,6021061,1990-07-01,1,67,31.808614,0.0,1.0,2.0,"[3,5,6]",2.0,-4.87,11017,139.0,80.0,142.0,88.0,53.0,52.000,NaN


In [15]:
df.to_csv('/mnt/disk2/dataset/ukb/hypertension.csv')

In [14]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
import os

from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, StringType, DoubleType

catalog = load_catalog(
    "default",
    **{
        "type": "rest",
        "uri": os.environ["ICEBERG_REST_URI"],
        "s3.endpoint": os.environ["ICEBERG_S3_ENDPOINT_PUBLIC"],
        "s3.access-key-id": os.environ["ICEBERG_S3_ACCESS_KEY_ID"],
        "s3.secret-access-key": os.environ["ICEBERG_S3_SECRET_ACCESS_KEY"],
    },
)


In [ ]:
import pyarrow as pa

cohort_info = client.get_cohort("record-J7Pj5Q0Jj70pZ16zQX7G9v9j")
arrow_schema = pa.Schema.from_pandas(df)
table = catalog.create_table_if_not_exists(
    identifier=f"ukb.{cohort_info.name.replace(' ', '_')}",
    schema=arrow_schema,
)
table.append(df=pa.Table.from_pandas(df))

In [23]:
table

hypertension_cohort(
  1: participant.eid: optional string,
  2: participant.p131286: optional string,
  3: participant.p31: optional long,
  4: participant.p21003_i0: optional long,
  5: participant.p21001_i0: optional double,
  6: participant.p1239_i0: optional double,
  7: participant.p1249_i0: optional double,
  8: participant.p20117_i0: optional double,
  9: participant.p6138_i0: optional string,
  10: participant.p738_i0: optional double,
  11: participant.p22189: optional double,
  12: participant.p54_i0: optional long,
  13: participant.p4080_i0_a0: optional double,
  14: participant.p4079_i0_a0: optional double,
  15: participant.p4080_i0_a1: optional double,
  16: participant.p4079_i0_a1: optional double,
  17: participant.p2966_i0: optional double,
  18: participant.p20161_i0: optional double,
  19: olink_instance_0.eid: optional string
),
partition by: [],
sort order: [],
snapshot: Operation.APPEND: id=1506258249351590317, schema_id=0